# Kaggle 독립 DL 캠페인\n\n비공개 Dataset에 `train.csv`와 `trackman_history.csv`를 연결하고 GPU를 켠 뒤 아래 코드 셀을 실행합니다. 기존 결과를 이어가려면 `independent_dl_campaign_v1` 폴더가 포함된 이전 Notebook Output 또는 비공개 Dataset도 함께 연결하세요.\n\n첫 캠페인은 여러 세션이 걸릴 수 있습니다. 세션 종료 전 **Save Version**으로 `/kaggle/working/independent_dl_campaign_v1`을 보존하세요. 성공 시 `INDEPENDENT_DL_CAMPAIGN_CHECKPOINTED`가 출력됩니다. 오류가 나면 첫 traceback부터 마지막 줄까지 보내주세요.\n

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys


INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
REPO_URL = "https://github.com/yh4752/lg-aimers-9th-competition.git"
REPO_DIR = WORKING_ROOT / "lg-aimers-9th-competition"
REQUIRED_CODE_COMMIT = "c36811956e632a00f775525f83bcb666ff2ec9d6"
RUNTIME_DIR = WORKING_ROOT / "independent_dl_runtime_v1"
CAMPAIGN_OUTPUT_DIR = WORKING_ROOT / "independent_dl_campaign_v1"


def run_checked(command, *, cwd=None, env=None):
    completed = subprocess.run(
        [str(item) for item in command],
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.returncode != 0:
        raise RuntimeError(
            f"명령 실패(returncode={completed.returncode}): {command[0]}"
        )
    return completed


def find_unique_file(filename):
    matches = sorted(path for path in INPUT_ROOT.rglob(filename) if path.is_file())
    if len(matches) != 1:
        detail = "\n".join(f"- {path}" for path in matches) or "- 발견되지 않음"
        reason = "여러 개 발견" if len(matches) > 1 else "찾지 못함"
        raise RuntimeError(
            f"{filename}을(를) 정확히 하나 찾아야 하지만 {reason}:\n{detail}"
        )
    return matches[0]


def find_checkpoint_source():
    matches = sorted(
        path
        for path in INPUT_ROOT.rglob("independent_dl_campaign_v1")
        if path.is_dir()
    )
    if len(matches) > 1:
        detail = "\n".join(f"- {path}" for path in matches)
        raise RuntimeError(
            "체크포인트 폴더를 여러 개 발견했습니다. 하나만 Dataset 입력으로 "
            f"연결하세요:\n{detail}"
        )
    return matches[0] if matches else None


if not INPUT_ROOT.is_dir() or not WORKING_ROOT.is_dir():
    raise RuntimeError("Kaggle Notebook 런타임에서 실행해야 합니다.")

train_path = find_unique_file("train.csv")
trackman_path = find_unique_file("trackman_history.csv")
if train_path.parent != trackman_path.parent:
    raise RuntimeError(
        "train.csv와 trackman_history.csv가 같은 Dataset 폴더에 있어야 합니다.\n"
        f"train: {train_path}\ntrackman: {trackman_path}"
    )
DATA_DIR = train_path.parent
print("공식 데이터:", DATA_DIR)

checkpoint_source = find_checkpoint_source()
if CAMPAIGN_OUTPUT_DIR.exists():
    if not CAMPAIGN_OUTPUT_DIR.is_dir():
        raise RuntimeError(
            f"기존 출력 경로가 디렉터리가 아닙니다: {CAMPAIGN_OUTPUT_DIR}"
        )
    print("기존 /kaggle/working 체크포인트를 그대로 사용합니다.")
elif checkpoint_source is not None:
    print("체크포인트 입력을 /kaggle/working으로 복사합니다:", checkpoint_source)
    shutil.copytree(
        checkpoint_source,
        CAMPAIGN_OUTPUT_DIR,
        dirs_exist_ok=False,
    )
else:
    CAMPAIGN_OUTPUT_DIR.mkdir(parents=False, exist_ok=False)
    print("연결된 체크포인트가 없어 새 캠페인으로 시작합니다.")

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"기존 경로가 Git 저장소가 아닙니다: {REPO_DIR}")
    run_checked(["git", "fetch", "--prune", "origin"], cwd=REPO_DIR)
else:
    run_checked(["git", "clone", "--no-checkout", REPO_URL, REPO_DIR])
    run_checked(["git", "fetch", "--prune", "origin"], cwd=REPO_DIR)
run_checked(["git", "checkout", "--detach", REQUIRED_CODE_COMMIT], cwd=REPO_DIR)

head = run_checked(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).stdout.strip()
if head != REQUIRED_CODE_COMMIT:
    raise RuntimeError(f"코드 커밋 불일치: {head}")

requirements = REPO_DIR / "experiments/independent_dl/requirements-colab.txt"
requirements_sha = hashlib.sha256(requirements.read_bytes()).hexdigest()
marker = RUNTIME_DIR / ".requirements_sha256"
if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != requirements_sha:
    if RUNTIME_DIR.exists():
        raise RuntimeError(
            "요구사항 해시가 다른 기존 런타임 폴더가 있습니다. 새 Kaggle 세션에서 "
            "다시 실행하세요."
        )
    RUNTIME_DIR.mkdir(parents=True)
    run_checked(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--target",
            str(RUNTIME_DIR),
            "--upgrade",
            "--no-deps",
            "-r",
            str(requirements),
        ]
    )
    marker.write_text(requirements_sha + "\n", encoding="utf-8")

child_env = os.environ.copy()
child_env["PYTHONPATH"] = os.pathsep.join(
    [str(RUNTIME_DIR), str(REPO_DIR), child_env.get("PYTHONPATH", "")]
)
child_env["PYTHONUNBUFFERED"] = "1"

gpu_check = (
    "import torch; "
    "assert torch.cuda.is_available(), 'CUDA GPU가 없습니다'; "
    "print('GPU:', torch.cuda.get_device_name(0))"
)
run_checked([sys.executable, "-c", gpu_check], cwd=REPO_DIR, env=child_env)

config = REPO_DIR / "experiments/independent_dl/configs/campaign_v1.json"
command = [
    sys.executable,
    "-m",
    "experiments.independent_dl.run_campaign",
    "run",
    "--config",
    str(config),
    "--data-dir",
    str(DATA_DIR),
    "--output-dir",
    str(CAMPAIGN_OUTPUT_DIR),
]
process = subprocess.Popen(
    command,
    cwd=REPO_DIR,
    env=child_env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(
        f"독립 DL 캠페인 자식 프로세스가 실패했습니다(returncode={returncode})."
    )

manifest = CAMPAIGN_OUTPUT_DIR / "campaign_manifest.json"
summary = CAMPAIGN_OUTPUT_DIR / "campaign_summary.json"
if not manifest.is_file():
    raise RuntimeError(f"campaign_manifest.json이 생성되지 않았습니다: {manifest}")
payload = json.loads(manifest.read_text(encoding="utf-8"))
completed = [
    candidate_id
    for candidate_id, entry in payload["candidates"].items()
    if entry["state"] == "completed"
]
print("INDEPENDENT_DL_CAMPAIGN_CHECKPOINTED")
print("Kaggle Save Version에 보존할 결과 루트:", CAMPAIGN_OUTPUT_DIR)
print("Manifest:", manifest)
print("Summary:", summary if summary.is_file() else "아직 없음")
print("완료 후보 수:", len(completed))
print("최근 완료 후보:", completed[-1] if completed else "아직 없음")
